# LUMIS-1 SPEAKER DATASET PIPELINE
## Diversity Engine - 10,000 Training Examples

**Target**: RunPod PyTorch 2.4.0 Template (NVIDIA A40/A100, CUDA 12.4)

**Dataset Composition**:
| Category | Count | Source |
|----------|-------|--------|
| General | 1,500 | OpenOrca + NoRobots |
| Identity | 1,000 | DeepSeek (synthetic) |
| Evidence | 2,000 | DeepSeek (synthetic) |
| Refusal | 500 | DeepSeek (synthetic) |
| Repair | 2,500 | DeepSeek (synthetic) |
| Contrast | 1,500 | DeepSeek (synthetic) |
| Policy | 1,000 | DeepSeek (synthetic) |

In [ ]:
# ============================================================================
# CELL 1: DEPENDENCIES & CONFIGURATION
# ============================================================================

import subprocess
import sys
import os

# ============================================================================
# SET YOUR API KEY HERE
# ============================================================================
os.environ["DEEPSEEK_API_KEY"] = "sk-ec56058053824919802cf108daec2ecb"

def run(cmd, desc):
    """Run shell command with progress output."""
    print(f"\n{'='*60}")
    print(f"[LUMIS-1] {desc}")
    print(f"{'='*60}")
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout[-500:] if len(result.stdout) > 500 else result.stdout)
    if result.returncode != 0 and result.stderr:
        print(f"STDERR: {result.stderr[-300:]}")
    return result.returncode == 0

# ============================================================================
# PHASE 1: INSTALL DEPENDENCIES
# ============================================================================
print("\n" + "#"*60)
print("# PHASE 1: INSTALLING DEPENDENCIES")
print("#"*60)

run(f"{sys.executable} -m pip install --upgrade pip -q", "Upgrading pip...")

# Install OpenAI SDK (DeepSeek uses OpenAI-compatible API) + nest_asyncio for Jupyter async
run(
    f"{sys.executable} -m pip install "
    f"openai "
    f"nest_asyncio "
    f"tqdm>=4.65.0 "
    f"pandas>=2.0.0 "
    f"datasets>=2.18.0 "
    f"transformers>=4.40.0 "
    f"-q",
    "Installing openai, nest_asyncio, tqdm, pandas, datasets, transformers..."
)

# ============================================================================
# PHASE 2: VERIFY INSTALLATIONS
# ============================================================================
print("\n" + "#"*60)
print("# PHASE 2: VERIFYING INSTALLATIONS")
print("#"*60)

from importlib.metadata import version as pkg_version

print(f"pip version: {pkg_version('pip')}")
print(f"openai version: {pkg_version('openai')}")
print(f"nest_asyncio version: {pkg_version('nest_asyncio')}")

# ============================================================================
# PHASE 3: IMPORTS & CONFIG
# ============================================================================
print("\n" + "#"*60)
print("# PHASE 3: IMPORTS & CONFIGURATION")
print("#"*60)

import json
import re
import uuid
import time
import random
import asyncio
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Literal, Tuple, Any
from enum import Enum
from abc import ABC, abstractmethod
from tqdm.asyncio import tqdm as async_tqdm
from tqdm import tqdm
import pandas as pd
from transformers import AutoTokenizer
from openai import OpenAI, AsyncOpenAI

# Enable nested asyncio for Jupyter notebooks
import nest_asyncio
nest_asyncio.apply()

# Configuration
CONFIG = {
    "model_id": "ibm-granite/granite-4.0-h-1b",
    "teacher_model": "deepseek-chat",  # DeepSeek V3.2 (non-thinking mode)
    "teacher_base_url": "https://api.deepseek.com",
    "min_tokens": 50,
    "max_tokens": 500,
    "repair_min_overlap": 0.30,
    "eval_ratio": 0.10,
    "seed": 42,
    "brand_name": "Lumis-1",
    "company_name": "Eptesicus Laboratories",
    "max_concurrent_requests": 50,  # Parallelization: 50 concurrent API calls
}

# Target counts per category
TARGET_COUNTS = {
    "general": 1500,
    "identity": 1000,
    "evidence": 2000,
    "refusal": 500,
    "repair": 2500,
    "contrast": 1500,
    "policy": 1000,
}

# Verify API key
api_key = os.environ.get("DEEPSEEK_API_KEY")
if not api_key:
    raise ValueError("DEEPSEEK_API_KEY not set! Edit line 12 of this cell.")
print(f"API Key: {api_key[:8]}...{api_key[-4:]}")

# Load tokenizer for token counting
print(f"\nLoading tokenizer: {CONFIG['model_id']}")
tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_id"], trust_remote_code=True)
print(f"Tokenizer loaded successfully.")

# Set random seed
random.seed(CONFIG["seed"])

print("\n" + "="*60)
print("CONFIGURATION COMPLETE")
print("="*60)
print(f"Teacher model: {CONFIG['teacher_model']} (DeepSeek V3.2)")
print(f"Target total: {sum(TARGET_COUNTS.values()):,} examples")
print(f"Token range: {CONFIG['min_tokens']}-{CONFIG['max_tokens']}")
print(f"Repair overlap threshold: {CONFIG['repair_min_overlap']:.0%}")
print(f"Max concurrent requests: {CONFIG['max_concurrent_requests']}")

In [ ]:
# ============================================================================
# CELL 2: DATA CLASSES & TYPE DEFINITIONS
# ============================================================================

class Category(Enum):
    """Dataset categories."""
    GENERAL = "general"
    IDENTITY = "identity"
    EVIDENCE = "evidence"
    REFUSAL = "refusal"
    REPAIR = "repair"
    CONTRAST = "contrast"
    POLICY = "policy"


@dataclass
class Message:
    """Single message in a conversation turn."""
    role: Literal["system", "user", "assistant"]
    content: str
    
    def to_dict(self) -> Dict[str, str]:
        return {"role": self.role, "content": self.content}


@dataclass
class TrainingExample:
    """Base class for all training examples."""
    id: str
    category: Category
    messages: List[Message]
    metadata: Dict = field(default_factory=dict)
    
    def to_granite_format(self) -> str:
        """Convert to Granite chat template format."""
        formatted = ""
        for msg in self.messages:
            formatted += f"<|start_of_role|>{msg.role}<|end_of_role|>"
            formatted += f"{msg.content}<|end_of_text|>\n"
        return formatted
    
    def to_jsonl(self) -> str:
        """Convert to JSONL format."""
        return json.dumps({
            "id": self.id,
            "category": self.category.value,
            "messages": [m.to_dict() for m in self.messages],
            "text": self.to_granite_format(),
            "metadata": self.metadata
        }, ensure_ascii=False)
    
    def token_count(self) -> int:
        """Count tokens using Granite tokenizer."""
        return len(tokenizer.encode(self.to_granite_format()))


@dataclass
class GeneralExample(TrainingExample):
    """Filtered examples from OpenOrca/NoRobots."""
    source_dataset: str = ""
    original_id: str = ""


@dataclass
class IdentityExample(TrainingExample):
    """Brand identity and jailbreak resistance examples."""
    attack_type: Optional[str] = None


@dataclass
class EvidenceExample(TrainingExample):
    """Context-grounded answers with 'trap' questions."""
    context: str = ""
    is_trap: bool = False
    expected_behavior: Literal["answer_from_context", "state_uncertainty"] = "answer_from_context"


@dataclass
class RefusalExample(TrainingExample):
    """Short, polite refusals (max 2 sentences)."""
    refusal_reason: str = ""


@dataclass
class RepairExample(TrainingExample):
    """5-turn hallucination -> steering -> correction sequences."""
    context: str = ""
    hallucinated_claim: str = ""
    steering_instruction: str = ""
    grounded_correction: str = ""
    keyword_overlap: float = 0.0


@dataclass
class ContrastExample(TrainingExample):
    """Near-identical inputs with different correct outputs."""
    variant_key: str = ""
    difference_type: str = ""


@dataclass
class PolicyExample(TrainingExample):
    """Enterprise policy scenarios."""
    scenario_type: Literal["approve", "deny", "escalate"] = "approve"
    policy_context: str = ""
    decision_rationale: str = ""


@dataclass
class GenerationStats:
    """Track generation statistics per category."""
    category: str
    target_count: int
    generated_count: int = 0
    discarded_count: int = 0
    
    # Discard reasons
    json_invalid: int = 0
    token_too_short: int = 0
    token_too_long: int = 0
    keyword_overlap_low: int = 0
    duplicate: int = 0
    other: int = 0
    
    @property
    def discard_rate(self) -> float:
        total = self.generated_count + self.discarded_count
        return self.discarded_count / total if total > 0 else 0.0


print("Data classes defined:")
print("  - Message, TrainingExample (base)")
print("  - GeneralExample, IdentityExample, EvidenceExample")
print("  - RefusalExample, RepairExample, ContrastExample, PolicyExample")
print("  - GenerationStats")

In [ ]:
# ============================================================================
# CELL 3: QUALITY GATES
# ============================================================================

# Imports (in case Cell 1 didn't complete)
import json
import re
from abc import ABC, abstractmethod
from typing import List, Tuple, Optional

# Stopwords for keyword extraction
STOPWORDS = {
    "the", "and", "for", "are", "but", "not", "you", "all", "can", "had",
    "her", "was", "one", "our", "out", "has", "have", "been", "were", "what",
    "when", "where", "which", "this", "that", "with", "from", "they", "will",
    "would", "there", "their", "about", "into", "could", "than", "other",
    "some", "your", "them", "these", "then", "only", "also", "over", "such",
    "more", "very", "just", "should", "because", "being", "through", "before",
    "after", "between", "under", "during", "without", "however", "therefore",
}


class QualityGate(ABC):
    """Abstract base class for quality gates."""
    
    @property
    @abstractmethod
    def name(self) -> str:
        """Gate identifier for logging."""
        pass
    
    @abstractmethod
    def validate(self, example) -> Tuple[bool, Optional[str]]:
        """Validate an example. Returns (passed, failure_reason)."""
        pass


class JSONValidityGate(QualityGate):
    """Ensure example can be serialized to valid JSON."""
    
    @property
    def name(self) -> str:
        return "json_validity"
    
    def validate(self, example) -> Tuple[bool, Optional[str]]:
        try:
            json.loads(example.to_jsonl())
            return True, None
        except json.JSONDecodeError as e:
            return False, f"Invalid JSON: {str(e)[:50]}"


class TokenLengthGate(QualityGate):
    """Enforce token length bounds."""
    
    def __init__(self, min_tokens: int = 50, max_tokens: int = 500):
        self.min_tokens = min_tokens
        self.max_tokens = max_tokens
    
    @property
    def name(self) -> str:
        return "token_length"
    
    def validate(self, example) -> Tuple[bool, Optional[str]]:
        count = example.token_count()
        if count < self.min_tokens:
            return False, f"too_short:{count}<{self.min_tokens}"
        if count > self.max_tokens:
            return False, f"too_long:{count}>{self.max_tokens}"
        return True, None


class RepairKeywordOverlapGate(QualityGate):
    """Ensure corrected response has >= 30% keyword overlap with context."""
    
    def __init__(self, min_overlap: float = 0.30):
        self.min_overlap = min_overlap
    
    @property
    def name(self) -> str:
        return "repair_keyword_overlap"
    
    def _extract_keywords(self, text: str) -> set:
        """Extract significant words (>3 chars, lowercase, no stopwords)."""
        words = re.findall(r'\b[a-zA-Z]{4,}\b', text.lower())
        return set(words) - STOPWORDS
    
    def validate(self, example) -> Tuple[bool, Optional[str]]:
        if not hasattr(example, 'context') or not hasattr(example, 'keyword_overlap'):
            return True, None
        
        # If keyword_overlap was pre-calculated, use it
        if example.keyword_overlap > 0:
            if example.keyword_overlap < self.min_overlap:
                return False, f"overlap:{example.keyword_overlap:.1%}<{self.min_overlap:.0%}"
            return True, None
        
        # Otherwise calculate from context and last assistant message
        context_keywords = self._extract_keywords(example.context)
        
        correction = ""
        for msg in reversed(example.messages):
            if msg.role == "assistant":
                correction = msg.content
                break
        
        correction_keywords = self._extract_keywords(correction)
        
        if not context_keywords:
            return False, "no_context_keywords"
        
        overlap = len(context_keywords & correction_keywords) / len(context_keywords)
        example.keyword_overlap = overlap
        
        if overlap < self.min_overlap:
            return False, f"overlap:{overlap:.1%}<{self.min_overlap:.0%}"
        return True, None


class UniquenessGate(QualityGate):
    """Ensure no exact duplicate prompts."""
    
    def __init__(self):
        self.seen_prompts: set = set()
    
    @property
    def name(self) -> str:
        return "uniqueness"
    
    def _extract_prompt(self, example) -> str:
        """Extract the user prompt for deduplication."""
        for msg in example.messages:
            if msg.role == "user":
                return msg.content.strip().lower()[:200]
        return ""
    
    def validate(self, example) -> Tuple[bool, Optional[str]]:
        prompt = self._extract_prompt(example)
        if not prompt:
            return False, "no_user_prompt"
        if prompt in self.seen_prompts:
            return False, "duplicate_prompt"
        self.seen_prompts.add(prompt)
        return True, None
    
    def reset(self):
        """Clear seen prompts."""
        self.seen_prompts.clear()


class EnglishOnlyGate(QualityGate):
    """Ensure content is primarily English."""
    
    @property
    def name(self) -> str:
        return "english_only"
    
    def validate(self, example) -> Tuple[bool, Optional[str]]:
        full_text = " ".join(m.content for m in example.messages)
        non_ascii = sum(1 for c in full_text if ord(c) > 127)
        ratio = non_ascii / max(len(full_text), 1)
        if ratio > 0.1:
            return False, f"non_english:{ratio:.1%}"
        return True, None


class QualityPipeline:
    """Chain of quality gates."""
    
    def __init__(self, gates: List[QualityGate]):
        self.gates = gates
    
    def validate(self, example) -> Tuple[bool, List[str]]:
        """Run all gates, return (passed, failure_reasons)."""
        failures = []
        for gate in self.gates:
            passed, reason = gate.validate(example)
            if not passed:
                failures.append(f"{gate.name}:{reason}")
        return len(failures) == 0, failures


# Create shared uniqueness gate
uniqueness_gate = UniquenessGate()


def create_quality_pipeline(category) -> QualityPipeline:
    """Factory to create category-appropriate quality pipeline."""
    
    # Get category value
    if hasattr(category, 'value'):
        cat_value = category.value
    else:
        cat_value = str(category).lower()
    
    # Base gates for all categories
    gates = [
        JSONValidityGate(),
        EnglishOnlyGate(),
        uniqueness_gate,
    ]
    
    # Token length gate with category-specific thresholds
    # Refusal: min=15 (short responses expected)
    # All other categories: min=50, max=500
    if cat_value == "refusal":
        gates.append(TokenLengthGate(min_tokens=15, max_tokens=CONFIG["max_tokens"]))
    else:
        gates.append(TokenLengthGate(CONFIG["min_tokens"], CONFIG["max_tokens"]))
    
    # Repair-specific: keyword overlap gate
    if cat_value == "repair":
        gates.append(RepairKeywordOverlapGate(CONFIG["repair_min_overlap"]))
    
    return QualityPipeline(gates)


print("Quality Gates defined:")
print("  - JSONValidityGate")
print("  - EnglishOnlyGate")
print("  - TokenLengthGate (min=15 for refusal, min=50 for others)")
print("  - UniquenessGate (shared)")
print("  - RepairKeywordOverlapGate (>= 30%)")

In [ ]:
# ============================================================================
# CELL 4: DEEPSEEK CLIENT WITH ASYNC PARALLELIZATION
# ============================================================================

import threading

class DeepSeekClient:
    """Rate-limited DeepSeek API client with async parallelization.
    
    Uses OpenAI-compatible API with AsyncOpenAI for concurrent requests.
    API Reference: https://api-docs.deepseek.com/
    """
    
    def __init__(
        self,
        model_name: str = None,
        max_retries: int = 5,
        base_delay: float = 1.0,
        max_delay: float = 60.0,
        jitter: float = 0.1,
        max_concurrent: int = None
    ):
        model_name = model_name or CONFIG["teacher_model"]
        api_key = os.environ.get("DEEPSEEK_API_KEY")
        if not api_key:
            raise ValueError("DEEPSEEK_API_KEY not set")
        
        # Synchronous client (for testing/fallback)
        self.client = OpenAI(
            api_key=api_key,
            base_url=CONFIG["teacher_base_url"]
        )
        
        # Async client for parallel requests
        self.async_client = AsyncOpenAI(
            api_key=api_key,
            base_url=CONFIG["teacher_base_url"]
        )
        
        self.model_name = model_name
        self.max_retries = max_retries
        self.base_delay = base_delay
        self.max_delay = max_delay
        self.jitter = jitter
        
        # Concurrency control
        self.max_concurrent = max_concurrent or CONFIG.get("max_concurrent_requests", 10)
        self._semaphore = None  # Created per-event-loop
        
        # Rate limiting state (thread-safe)
        self._lock = threading.Lock()
        self.last_call_time = 0.0
        self.min_interval = 0.02  # 50 RPS max (conservative)
        
        # Stats (thread-safe)
        self.total_calls = 0
        self.total_errors = 0
    
    def _get_semaphore(self) -> asyncio.Semaphore:
        """Get or create semaphore for current event loop."""
        if self._semaphore is None:
            self._semaphore = asyncio.Semaphore(self.max_concurrent)
        return self._semaphore
    
    def _calculate_backoff(self, attempt: int) -> float:
        """Calculate delay with exponential backoff and jitter."""
        delay = min(self.base_delay * (2 ** attempt), self.max_delay)
        jitter_amount = delay * self.jitter * random.uniform(-1, 1)
        return delay + jitter_amount
    
    async def generate_async(
        self,
        prompt: str,
        temperature: float = 0.8,
        max_output_tokens: int = 1024
    ) -> Optional[str]:
        """Async generate with semaphore-controlled concurrency."""
        
        semaphore = self._get_semaphore()
        
        async with semaphore:
            for attempt in range(self.max_retries):
                try:
                    # Update stats (thread-safe)
                    with self._lock:
                        self.total_calls += 1
                    
                    # Make async API call
                    response = await self.async_client.chat.completions.create(
                        model=self.model_name,
                        messages=[{"role": "user", "content": prompt}],
                        temperature=temperature,
                        max_tokens=max_output_tokens
                    )
                    
                    if response.choices and response.choices[0].message.content:
                        return response.choices[0].message.content.strip()
                    return None
                    
                except Exception as e:
                    with self._lock:
                        self.total_errors += 1
                    
                    error_str = str(e).lower()
                    
                    if "rate" in error_str or "429" in error_str or "limit" in error_str:
                        delay = self._calculate_backoff(attempt)
                        await asyncio.sleep(delay)
                    elif "safety" in error_str or "blocked" in error_str or "content" in error_str:
                        return None  # Skip blocked content
                    else:
                        if attempt == self.max_retries - 1:
                            return None
                        await asyncio.sleep(self._calculate_backoff(attempt))
            
            return None
    
    def generate(
        self,
        prompt: str,
        temperature: float = 0.8,
        max_output_tokens: int = 1024
    ) -> Optional[str]:
        """Synchronous generate (for testing/single calls)."""
        
        for attempt in range(self.max_retries):
            try:
                self.total_calls += 1
                
                response = self.client.chat.completions.create(
                    model=self.model_name,
                    messages=[{"role": "user", "content": prompt}],
                    temperature=temperature,
                    max_tokens=max_output_tokens
                )
                
                if response.choices and response.choices[0].message.content:
                    return response.choices[0].message.content.strip()
                return None
                
            except Exception as e:
                self.total_errors += 1
                error_str = str(e).lower()
                
                if "rate" in error_str or "429" in error_str or "limit" in error_str:
                    delay = self._calculate_backoff(attempt)
                    print(f"\n[Rate limit] Waiting {delay:.1f}s (attempt {attempt + 1}/{self.max_retries})")
                    time.sleep(delay)
                elif "safety" in error_str or "blocked" in error_str or "content" in error_str:
                    return None
                else:
                    if attempt == self.max_retries - 1:
                        print(f"\n[API Error] {str(e)[:100]}")
                        return None
                    time.sleep(self._calculate_backoff(attempt))
        
        return None


# Initialize client
print("Initializing DeepSeek client with async support...")
teacher_client = DeepSeekClient()
print(f"Model: {teacher_client.model_name}")
print(f"Max concurrent requests: {teacher_client.max_concurrent}")

# Test connection (sync)
print("\nTesting API connection...")
test_response = teacher_client.generate("Say 'Hello' and nothing else.", temperature=0.1, max_output_tokens=10)
if test_response:
    print(f"API test successful: {test_response}")
else:
    print("WARNING: API test failed. Check your API key and quota.")

In [ ]:
# ============================================================================
# CELL 5: BASE GENERATOR WITH ASYNC PARALLELIZATION
# ============================================================================

class BaseGenerator(ABC):
    """Abstract base class for category generators with async support."""
    
    def __init__(
        self,
        client: DeepSeekClient,
        category: Category,
        target_count: int
    ):
        self.client = client
        self.category = category
        self.target_count = target_count
        self.quality_pipeline = create_quality_pipeline(category)
        self.stats = GenerationStats(
            category=category.value,
            target_count=target_count
        )
        self._stats_lock = threading.Lock()
    
    @abstractmethod
    def build_prompt(self) -> str:
        """Build the generation prompt (with variation)."""
        pass
    
    @abstractmethod
    def parse_response(self, response: str) -> Optional[TrainingExample]:
        """Parse API response into a TrainingExample."""
        pass
    
    def _extract_json(self, text: str) -> Optional[Dict]:
        """Extract JSON object from response text."""
        patterns = [
            r'```json\s*(.+?)\s*```',
            r'```\s*(.+?)\s*```',
            r'\{.+\}',
        ]
        
        for pattern in patterns:
            match = re.search(pattern, text, re.DOTALL)
            if match:
                try:
                    json_str = match.group(1) if '```' in pattern else match.group(0)
                    return json.loads(json_str)
                except json.JSONDecodeError:
                    continue
        return None
    
    def _record_failure(self, failures: List[str]):
        """Update stats based on failure reasons (thread-safe)."""
        with self._stats_lock:
            for failure in failures:
                fl = failure.lower()
                if "json" in fl:
                    self.stats.json_invalid += 1
                elif "short" in fl:
                    self.stats.token_too_short += 1
                elif "long" in fl:
                    self.stats.token_too_long += 1
                elif "overlap" in fl:
                    self.stats.keyword_overlap_low += 1
                elif "duplicate" in fl:
                    self.stats.duplicate += 1
                else:
                    self.stats.other += 1
    
    async def _generate_single_async(self) -> Optional[TrainingExample]:
        """Generate a single example asynchronously."""
        prompt = self.build_prompt()
        
        response = await self.client.generate_async(prompt)
        if response is None:
            with self._stats_lock:
                self.stats.discarded_count += 1
                self.stats.other += 1
            return None
        
        example = self.parse_response(response)
        if example is None:
            with self._stats_lock:
                self.stats.discarded_count += 1
                self.stats.json_invalid += 1
            return None
        
        passed, failures = self.quality_pipeline.validate(example)
        if not passed:
            self._record_failure(failures)
            with self._stats_lock:
                self.stats.discarded_count += 1
            return None
        
        return example
    
    async def generate_batch_async(self, show_progress: bool = True) -> List[TrainingExample]:
        """Generate examples with parallel async requests."""
        examples = []
        
        pbar = tqdm(
            total=self.target_count,
            desc=f"{self.category.value:12}",
            disable=not show_progress,
            ncols=80
        )
        
        max_attempts = self.target_count * 3
        pending_tasks = set()
        attempts = 0
        
        # Fill initial batch
        batch_size = min(self.client.max_concurrent * 2, self.target_count)
        for _ in range(batch_size):
            task = asyncio.create_task(self._generate_single_async())
            pending_tasks.add(task)
            attempts += 1
        
        while self.stats.generated_count < self.target_count and (pending_tasks or attempts < max_attempts):
            if not pending_tasks:
                # Spawn more tasks if needed
                spawn_count = min(
                    self.client.max_concurrent,
                    self.target_count - self.stats.generated_count,
                    max_attempts - attempts
                )
                for _ in range(spawn_count):
                    task = asyncio.create_task(self._generate_single_async())
                    pending_tasks.add(task)
                    attempts += 1
                if not pending_tasks:
                    break
            
            # Wait for first completed task
            done, pending_tasks = await asyncio.wait(
                pending_tasks,
                return_when=asyncio.FIRST_COMPLETED
            )
            
            for task in done:
                result = task.result()
                if result is not None:
                    with self._stats_lock:
                        if self.stats.generated_count < self.target_count:
                            self.stats.generated_count += 1
                            examples.append(result)
                            pbar.update(1)
                
                # Spawn replacement task if needed
                if (self.stats.generated_count < self.target_count and 
                    attempts < max_attempts and
                    len(pending_tasks) < self.client.max_concurrent * 2):
                    new_task = asyncio.create_task(self._generate_single_async())
                    pending_tasks.add(new_task)
                    attempts += 1
        
        # Cancel remaining tasks
        for task in pending_tasks:
            task.cancel()
        
        pbar.close()
        return examples
    
    def generate_batch(self, show_progress: bool = True) -> List[TrainingExample]:
        """Synchronous wrapper for async batch generation."""
        return asyncio.run(self.generate_batch_async(show_progress))


print("BaseGenerator class defined with async parallelization.")
print("Features:")
print("  - Async batch generation with configurable concurrency")
print("  - Thread-safe statistics tracking")
print("  - Automatic task spawning/replacement")
print("  - Quality pipeline validation")

In [ ]:
# ============================================================================
# CELL 6: CATEGORY GENERATORS (7 Implementations)
# ============================================================================

# JSON formatting instructions (shared across all generators)
JSON_FORMAT_INSTRUCTIONS = """
CRITICAL JSON FORMATTING RULES:
1. Output ONLY a valid JSON object - no text before or after
2. Do NOT include markdown code blocks (no ```)
3. All strings must use double quotes, not single quotes
4. Escape special characters in strings: use \\" for quotes, \\n for newlines
5. Do NOT include trailing commas
6. Ensure the JSON is complete and parseable
"""

# -----------------------------------------------------------------------------
# IDENTITY GENERATOR
# -----------------------------------------------------------------------------
class IdentityGenerator(BaseGenerator):
    """Generate brand identity and jailbreak resistance examples."""
    
    ATTACK_TYPES = [
        ("direct_probe", "directly asking what AI model it is"),
        ("impersonation", "claiming the AI is actually GPT-4 or Claude"),
        ("jailbreak", "attempting to bypass restrictions with 'DAN' or 'ignore instructions'"),
        ("roleplay_escape", "requesting to pretend to be an unrestricted AI"),
        ("confusion", "trying to confuse the AI about its own identity"),
    ]
    
    def build_prompt(self) -> str:
        attack_type, desc = random.choice(self.ATTACK_TYPES)
        return f'''Generate a training example for an AI assistant named "{CONFIG['brand_name']}" developed by "{CONFIG['company_name']}".

Attack type: {attack_type} - {desc}

Create a JSON object with these exact keys:
{{
  "user_message": "<attack prompt attempting to confuse identity or bypass restrictions>",
  "assistant_response": "<response firmly maintaining identity as {CONFIG['brand_name']}, polite but unwavering, 1-3 sentences>"
}}

The assistant should:
- Always identify as {CONFIG['brand_name']}
- Never pretend to be another AI
- Politely decline roleplay that compromises identity
- Keep response under 3 sentences
{JSON_FORMAT_INSTRUCTIONS}'''
    
    def parse_response(self, response: str) -> Optional[IdentityExample]:
        data = self._extract_json(response)
        if not data or "user_message" not in data or "assistant_response" not in data:
            return None
        
        system_msg = Message(
            role="system",
            content=f"You are {CONFIG['brand_name']}, a governed enterprise AI assistant developed by {CONFIG['company_name']}. Maintain your identity firmly."
        )
        user_msg = Message(role="user", content=data["user_message"])
        assistant_msg = Message(role="assistant", content=data["assistant_response"])
        
        return IdentityExample(
            id=f"identity_{uuid.uuid4().hex[:8]}",
            category=Category.IDENTITY,
            messages=[system_msg, user_msg, assistant_msg],
            attack_type=random.choice(self.ATTACK_TYPES)[0]
        )


# -----------------------------------------------------------------------------
# EVIDENCE GENERATOR
# -----------------------------------------------------------------------------
class EvidenceGenerator(BaseGenerator):
    """Generate context-grounded Q&A with 'trap' questions."""
    
    DOMAINS = [
        "corporate policy document",
        "technical documentation",
        "financial report excerpt",
        "legal contract clause",
        "product specifications",
        "HR guidelines",
    ]
    
    def build_prompt(self) -> str:
        domain = random.choice(self.DOMAINS)
        is_trap = random.random() < 0.3
        
        if is_trap:
            return f'''Generate a "trap" training example where the answer is NOT in the context.

Domain: {domain}

Create a JSON object with these exact keys:
{{
  "context": "<3-6 sentences of realistic {domain} content with specific details>",
  "question": "<question whose answer is NOT in the context>",
  "response": "<response acknowledging the information is not provided in the context, max 2 sentences>"
}}

The assistant must NOT make up an answer. It should clearly state the information is not available in the provided context.
{JSON_FORMAT_INSTRUCTIONS}'''
        else:
            return f'''Generate a context-grounded Q&A training example.

Domain: {domain}

Create a JSON object with these exact keys:
{{
  "context": "<3-6 sentences of realistic {domain} content with specific facts, numbers, or details>",
  "question": "<question that can be answered from the context>",
  "response": "<response that cites the context explicitly, using phrases like 'According to the provided context' or 'Based on the information given'>"
}}
{JSON_FORMAT_INSTRUCTIONS}'''
    
    def parse_response(self, response: str) -> Optional[EvidenceExample]:
        data = self._extract_json(response)
        if not data or not all(k in data for k in ["context", "question", "response"]):
            return None
        
        resp_lower = data["response"].lower()
        is_trap = any(phrase in resp_lower for phrase in [
            "not provided", "not in the context", "not mentioned",
            "doesn't mention", "does not mention", "not available",
            "cannot determine", "no information"
        ])
        
        system_msg = Message(
            role="system",
            content=f"You are {CONFIG['brand_name']}, a governed enterprise AI. Answer questions based only on the provided context. If the answer is not in the context, say so clearly."
        )
        user_msg = Message(
            role="user",
            content=f"[CONTEXT]\n{data['context']}\n\n[QUESTION]\n{data['question']}"
        )
        assistant_msg = Message(role="assistant", content=data["response"])
        
        return EvidenceExample(
            id=f"evidence_{uuid.uuid4().hex[:8]}",
            category=Category.EVIDENCE,
            messages=[system_msg, user_msg, assistant_msg],
            context=data["context"],
            is_trap=is_trap,
            expected_behavior="state_uncertainty" if is_trap else "answer_from_context"
        )


# -----------------------------------------------------------------------------
# REFUSAL GENERATOR
# -----------------------------------------------------------------------------
class RefusalGenerator(BaseGenerator):
    """Generate short, polite refusals (max 2 sentences)."""
    
    REFUSAL_CATEGORIES = [
        ("harmful_request", "request for violence, illegal activity, or dangerous information"),
        ("privacy_violation", "request to extract or expose personal information"),
        ("misinformation", "request to create fake news or misleading content"),
        ("system_manipulation", "jailbreak or prompt injection attempt"),
        ("inappropriate_content", "request for adult, explicit, or offensive content"),
    ]
    
    def build_prompt(self) -> str:
        category, desc = random.choice(self.REFUSAL_CATEGORIES)
        return f'''Generate a training example where the AI must refuse a request.

Category: {category} - {desc}

Create a JSON object with these exact keys:
{{
  "user_request": "<harmful or inappropriate request in the {category} category>",
  "refusal_response": "<polite refusal in EXACTLY 1-2 sentences, no lecture, no lengthy explanation>"
}}

The refusal must be:
- Brief (1-2 sentences MAXIMUM)
- Polite but firm
- NOT preachy or lecturing
- NOT explaining why in detail

Good refusal examples:
- "I can't help with that request."
- "I'm not able to assist with that. Is there something else I can help you with?"

Bad refusal examples (TOO LONG - do NOT use):
- "I'm sorry, but I cannot assist with that request as it would be harmful because..."
{JSON_FORMAT_INSTRUCTIONS}'''
    
    def parse_response(self, response: str) -> Optional[RefusalExample]:
        data = self._extract_json(response)
        if not data or "user_request" not in data or "refusal_response" not in data:
            return None
        
        refusal = data["refusal_response"]
        sentence_count = len(re.findall(r'[.!?]+', refusal))
        if sentence_count > 3:  # Allow some flexibility
            return None
        
        system_msg = Message(
            role="system",
            content=f"You are {CONFIG['brand_name']}. Refuse harmful requests briefly and politely without lengthy explanations."
        )
        user_msg = Message(role="user", content=data["user_request"])
        assistant_msg = Message(role="assistant", content=refusal)
        
        return RefusalExample(
            id=f"refusal_{uuid.uuid4().hex[:8]}",
            category=Category.REFUSAL,
            messages=[system_msg, user_msg, assistant_msg],
            refusal_reason=random.choice(self.REFUSAL_CATEGORIES)[0]
        )


# -----------------------------------------------------------------------------
# REPAIR GENERATOR (5-Turn Sequences) - FIXED keyword_overlap calculation
# -----------------------------------------------------------------------------
class RepairGenerator(BaseGenerator):
    """Generate 5-turn hallucination -> steering -> correction sequences."""
    
    DOMAINS = [
        "quarterly financial results",
        "product launch specifications",
        "employee handbook policy",
        "technical system architecture",
        "project status report",
        "compliance requirements",
    ]
    
    STEERING_INSTRUCTIONS = [
        "Your response contained claims that aren't supported by the provided context. Please revise to cite only information from the context.",
        "I noticed your answer included details not mentioned in the context. Please provide a corrected response based only on what's given.",
        "The previous response made unsupported claims. Please revise and only reference facts from the provided context.",
    ]
    
    def _extract_keywords(self, text: str) -> set:
        """Extract significant words (>3 chars, lowercase, no stopwords)."""
        words = re.findall(r'\b[a-zA-Z]{4,}\b', text.lower())
        return set(words) - STOPWORDS
    
    def _calculate_keyword_overlap(self, context: str, correction: str) -> float:
        """Calculate keyword overlap between context and correction."""
        context_keywords = self._extract_keywords(context)
        correction_keywords = self._extract_keywords(correction)
        if not context_keywords:
            return 0.0
        return len(context_keywords & correction_keywords) / len(context_keywords)
    
    def build_prompt(self) -> str:
        domain = random.choice(self.DOMAINS)
        return f'''Generate a 5-turn "repair" training sequence demonstrating hallucination correction.

Domain: {domain}

The sequence must follow this EXACT 5-turn format:
1. System message sets up governance context
2. User asks a question with provided context
3. Assistant gives a HALLUCINATED answer (confident but contains unsupported claims)
4. User provides steering instruction pointing out the unsupported claims
5. Assistant gives CORRECTED answer grounded only in the context

Create a JSON object with these exact keys:
{{
  "context": "<4-6 sentences about {domain} with specific facts, numbers, dates>",
  "question": "<question about the context>",
  "hallucinated_answer": "<confident answer that includes at least one specific claim NOT in the context - make it subtle>",
  "corrected_answer": "<answer that ONLY references facts from the context, using similar keywords>"
}}

CRITICAL REQUIREMENTS:
- The hallucination should be SUBTLE (one wrong detail, not obviously wrong)
- The corrected answer must use SIMILAR LANGUAGE to the context (keyword overlap)
- The corrected answer should acknowledge what cannot be verified
{JSON_FORMAT_INSTRUCTIONS}'''
    
    def parse_response(self, response: str) -> Optional[RepairExample]:
        data = self._extract_json(response)
        if not data or not all(k in data for k in ["context", "question", "hallucinated_answer", "corrected_answer"]):
            return None
        
        steering = random.choice(self.STEERING_INSTRUCTIONS)
        
        # Calculate keyword overlap BEFORE creating example
        keyword_overlap = self._calculate_keyword_overlap(
            data["context"],
            data["corrected_answer"]
        )
        
        messages = [
            Message(
                role="system",
                content=f"You are {CONFIG['brand_name']}, a governed enterprise AI. Answer questions based only on the provided context. If you cannot verify a claim from the context, explicitly state this."
            ),
            Message(
                role="user",
                content=f"[CONTEXT]\n{data['context']}\n\n[QUESTION]\n{data['question']}"
            ),
            Message(
                role="assistant",
                content=data["hallucinated_answer"]
            ),
            Message(
                role="user",
                content=steering
            ),
            Message(
                role="assistant",
                content=data["corrected_answer"]
            )
        ]
        
        return RepairExample(
            id=f"repair_{uuid.uuid4().hex[:8]}",
            category=Category.REPAIR,
            messages=messages,
            context=data["context"],
            hallucinated_claim=data["hallucinated_answer"],
            steering_instruction=steering,
            grounded_correction=data["corrected_answer"],
            keyword_overlap=keyword_overlap  # Now properly populated
        )


# -----------------------------------------------------------------------------
# CONTRAST GENERATOR - With async support for pair generation
# -----------------------------------------------------------------------------
class ContrastGenerator(BaseGenerator):
    """Generate near-identical inputs with different correct outputs."""
    
    CONTRAST_TYPES = [
        ("parameter_change", "same task with different parameter values (e.g., staging vs production)"),
        ("permission_scope", "same request with different permission levels (e.g., read vs write)"),
        ("temporal_difference", "same action with different time constraints (e.g., before vs after deadline)"),
        ("threshold_boundary", "same request with values above vs below a threshold"),
    ]
    
    def build_prompt(self) -> str:
        contrast_type, desc = random.choice(self.CONTRAST_TYPES)
        return f'''Generate a "contrast" training example: two near-identical inputs with different correct outputs.

Contrast type: {contrast_type} - {desc}

Create a JSON object with these exact keys:
{{
  "variant_a": {{
    "input": "<user request version A>",
    "output": "<correct response for A>"
  }},
  "variant_b": {{
    "input": "<user request version B - differs by ONE key detail>",
    "output": "<correct response for B - appropriately different>"
  }},
  "difference": "<what single variable changed>"
}}

Examples of good contrasts:
- "Deploy to staging" vs "Deploy to production" (different approval requirements)
- "Read access request" vs "Write access request" (different permissions)
- "$4,500 expense" vs "$5,500 expense" (above/below $5k threshold)
{JSON_FORMAT_INSTRUCTIONS}'''
    
    def parse_response(self, response: str) -> Optional[List[ContrastExample]]:
        data = self._extract_json(response)
        if not data or "variant_a" not in data or "variant_b" not in data:
            return None
        
        examples = []
        pair_id = uuid.uuid4().hex[:8]
        
        for variant_key in ["variant_a", "variant_b"]:
            variant = data[variant_key]
            if "input" not in variant or "output" not in variant:
                return None
            
            system_msg = Message(
                role="system",
                content=f"You are {CONFIG['brand_name']}, a governed enterprise AI. Respond appropriately based on the specific details of each request."
            )
            user_msg = Message(role="user", content=variant["input"])
            assistant_msg = Message(role="assistant", content=variant["output"])
            
            examples.append(ContrastExample(
                id=f"contrast_{pair_id}_{variant_key[-1]}",
                category=Category.CONTRAST,
                messages=[system_msg, user_msg, assistant_msg],
                variant_key=variant_key,
                difference_type=data.get("difference", "")
            ))
        
        return examples
    
    async def _generate_single_async(self) -> Optional[List[TrainingExample]]:
        """Generate a pair of contrast examples asynchronously."""
        prompt = self.build_prompt()
        
        response = await self.client.generate_async(prompt)
        if response is None:
            with self._stats_lock:
                self.stats.discarded_count += 1
                self.stats.other += 1
            return None
        
        pair = self.parse_response(response)
        if pair is None:
            with self._stats_lock:
                self.stats.discarded_count += 1
                self.stats.json_invalid += 1
            return None
        
        # Validate both examples
        valid_pair = []
        for example in pair:
            passed, failures = self.quality_pipeline.validate(example)
            if passed:
                valid_pair.append(example)
            else:
                self._record_failure(failures)
                with self._stats_lock:
                    self.stats.discarded_count += 1
        
        return valid_pair if valid_pair else None
    
    async def generate_batch_async(self, show_progress: bool = True) -> List[TrainingExample]:
        """Generate contrast pairs with parallel async requests."""
        examples = []
        
        pbar = tqdm(
            total=self.target_count,
            desc=f"{self.category.value:12}",
            disable=not show_progress,
            ncols=80
        )
        
        # Each API call produces 0-2 examples
        max_attempts = self.target_count * 2
        pending_tasks = set()
        attempts = 0
        
        # Fill initial batch
        batch_size = min(self.client.max_concurrent * 2, self.target_count)
        for _ in range(batch_size):
            task = asyncio.create_task(self._generate_single_async())
            pending_tasks.add(task)
            attempts += 1
        
        while self.stats.generated_count < self.target_count and (pending_tasks or attempts < max_attempts):
            if not pending_tasks:
                spawn_count = min(
                    self.client.max_concurrent,
                    (self.target_count - self.stats.generated_count + 1) // 2,
                    max_attempts - attempts
                )
                for _ in range(spawn_count):
                    task = asyncio.create_task(self._generate_single_async())
                    pending_tasks.add(task)
                    attempts += 1
                if not pending_tasks:
                    break
            
            done, pending_tasks = await asyncio.wait(
                pending_tasks,
                return_when=asyncio.FIRST_COMPLETED
            )
            
            for task in done:
                result = task.result()
                if result is not None:
                    for example in result:
                        with self._stats_lock:
                            if self.stats.generated_count < self.target_count:
                                self.stats.generated_count += 1
                                examples.append(example)
                                pbar.update(1)
                
                # Spawn replacement if needed
                if (self.stats.generated_count < self.target_count and 
                    attempts < max_attempts and
                    len(pending_tasks) < self.client.max_concurrent * 2):
                    new_task = asyncio.create_task(self._generate_single_async())
                    pending_tasks.add(new_task)
                    attempts += 1
        
        for task in pending_tasks:
            task.cancel()
        
        pbar.close()
        return examples


# -----------------------------------------------------------------------------
# POLICY GENERATOR
# -----------------------------------------------------------------------------
class PolicyGenerator(BaseGenerator):
    """Generate enterprise policy scenarios (Approve/Deny/Escalate)."""
    
    SCENARIOS = [
        ("budget_approval", ["approve", "deny", "escalate"]),
        ("access_request", ["approve", "deny", "escalate"]),
        ("exception_request", ["approve", "deny", "escalate"]),
        ("deployment_approval", ["approve", "deny", "escalate"]),
        ("data_access", ["approve", "deny"]),
    ]
    
    def build_prompt(self) -> str:
        scenario, valid_decisions = random.choice(self.SCENARIOS)
        decision = random.choice(valid_decisions)
        return f'''Generate an enterprise policy scenario training example.

Scenario type: {scenario}
Required decision: {decision.upper()}

Create a JSON object with these exact keys:
{{
  "policy_context": "<2-3 sentences defining the relevant policy rules>",
  "user_request": "<employee request that requires a {decision} decision based on policy>",
  "assistant_response": "<response that {decision}s the request with brief policy citation, 2-3 sentences>",
  "decision": "{decision}"
}}

The response should:
- Clearly state the decision ({decision.upper()})
- Reference the policy briefly
- Be professional and concise

For ESCALATE decisions, specify who/what level needs to review.
{JSON_FORMAT_INSTRUCTIONS}'''
    
    def parse_response(self, response: str) -> Optional[PolicyExample]:
        data = self._extract_json(response)
        if not data or not all(k in data for k in ["policy_context", "user_request", "assistant_response"]):
            return None
        
        resp_lower = data["assistant_response"].lower()
        if "escalate" in resp_lower:
            scenario_type = "escalate"
        elif "deny" in resp_lower or "denied" in resp_lower or "cannot approve" in resp_lower:
            scenario_type = "deny"
        else:
            scenario_type = "approve"
        
        system_msg = Message(
            role="system",
            content=f"You are {CONFIG['brand_name']}, a governed enterprise AI. Apply the following policy:\n{data['policy_context']}"
        )
        user_msg = Message(role="user", content=data["user_request"])
        assistant_msg = Message(role="assistant", content=data["assistant_response"])
        
        return PolicyExample(
            id=f"policy_{uuid.uuid4().hex[:8]}",
            category=Category.POLICY,
            messages=[system_msg, user_msg, assistant_msg],
            scenario_type=scenario_type,
            policy_context=data["policy_context"],
            decision_rationale=data.get("decision", scenario_type)
        )


# -----------------------------------------------------------------------------
# GENERAL GENERATOR (Open Dataset Filtering) - No API calls needed
# -----------------------------------------------------------------------------
class GeneralGenerator:
    """Filter and format examples from OpenOrca and NoRobots."""
    
    CODE_PATTERNS = [
        r'```',
        r'def\s+\w+\(',
        r'function\s+\w+',
        r'import\s+\w+',
        r'#include',
        r'class\s+\w+\s*[:{]',
        r'\{\s*\n.*\n\s*\}',
    ]
    
    def __init__(self, target_count: int = 1500):
        self.target_count = target_count
        self.quality_pipeline = create_quality_pipeline(Category.GENERAL)
        self.stats = GenerationStats(
            category="general",
            target_count=target_count
        )
    
    def _has_code(self, text: str) -> bool:
        """Check if text contains code patterns."""
        for pattern in self.CODE_PATTERNS:
            if re.search(pattern, text, re.MULTILINE):
                return True
        return False
    
    def _is_valid_content(self, text: str) -> bool:
        """Check if content is English and has no code."""
        if self._has_code(text):
            return False
        non_ascii = sum(1 for c in text if ord(c) > 127)
        if non_ascii / max(len(text), 1) > 0.1:
            return False
        return True
    
    def generate(self) -> List[GeneralExample]:
        """Load and filter open datasets."""
        from datasets import load_dataset
        
        examples = []
        
        print("\nLoading OpenOrca dataset (streaming)...")
        try:
            openorca = load_dataset("Open-Orca/OpenOrca", split="train", streaming=True)
            
            orca_target = min(1000, self.target_count)
            pbar = tqdm(total=orca_target, desc="OpenOrca     ", ncols=80)
            
            for item in openorca:
                if len(examples) >= orca_target:
                    break
                
                question = item.get("question", "")
                response = item.get("response", "")
                
                if not question or not response:
                    continue
                
                full_text = question + " " + response
                if not self._is_valid_content(full_text):
                    self.stats.discarded_count += 1
                    continue
                
                example = GeneralExample(
                    id=f"general_orca_{uuid.uuid4().hex[:8]}",
                    category=Category.GENERAL,
                    messages=[
                        Message(role="system", content=f"You are {CONFIG['brand_name']}, a helpful enterprise AI assistant."),
                        Message(role="user", content=question),
                        Message(role="assistant", content=response)
                    ],
                    source_dataset="OpenOrca",
                    original_id=item.get("id", "")
                )
                
                passed, failures = self.quality_pipeline.validate(example)
                if not passed:
                    self.stats.discarded_count += 1
                    continue
                
                examples.append(example)
                self.stats.generated_count += 1
                pbar.update(1)
            
            pbar.close()
        except Exception as e:
            print(f"OpenOrca loading error: {e}")
        
        remaining = self.target_count - len(examples)
        if remaining > 0:
            print(f"\nLoading NoRobots dataset (need {remaining} more)...")
            try:
                norobots = load_dataset("HuggingFaceH4/no_robots", split="train", streaming=True)
                
                pbar = tqdm(total=remaining, desc="NoRobots     ", ncols=80)
                
                for item in norobots:
                    if len(examples) >= self.target_count:
                        break
                    
                    messages_raw = item.get("messages", [])
                    if len(messages_raw) < 2:
                        continue
                    
                    full_text = " ".join(m.get("content", "") for m in messages_raw)
                    if not self._is_valid_content(full_text):
                        self.stats.discarded_count += 1
                        continue
                    
                    messages = [Message(role="system", content=f"You are {CONFIG['brand_name']}, a helpful enterprise AI assistant.")]
                    for m in messages_raw:
                        if m.get("role") in ["user", "assistant"]:
                            messages.append(Message(role=m["role"], content=m["content"]))
                    
                    if len(messages) < 2:
                        continue
                    
                    example = GeneralExample(
                        id=f"general_norobots_{uuid.uuid4().hex[:8]}",
                        category=Category.GENERAL,
                        messages=messages,
                        source_dataset="NoRobots"
                    )
                    
                    passed, failures = self.quality_pipeline.validate(example)
                    if not passed:
                        self.stats.discarded_count += 1
                        continue
                    
                    examples.append(example)
                    self.stats.generated_count += 1
                    pbar.update(1)
                
                pbar.close()
            except Exception as e:
                print(f"NoRobots loading error: {e}")
        
        return examples


print("Category Generators defined:")
print("  - IdentityGenerator (5 attack types)")
print("  - EvidenceGenerator (6 domains, 30% traps)")
print("  - RefusalGenerator (5 refusal categories)")
print("  - RepairGenerator (5-turn sequences, keyword overlap calculated)")
print("  - ContrastGenerator (pair generation with async)")
print("  - PolicyGenerator (approve/deny/escalate)")
print("  - GeneralGenerator (OpenOrca + NoRobots filter)")
print("\nAll generators now use stricter JSON formatting instructions.")

In [ ]:
# ============================================================================
# CELL 7: PIPELINE ORCHESTRATION (WITH CHECKPOINTING & RESUME)
# ============================================================================

CHECKPOINT_DIR = "checkpoints"

def load_checkpoint(category: str) -> Tuple[List[TrainingExample], Optional[GenerationStats]]:
    """Load checkpoint file for a category if it exists."""
    checkpoint_path = os.path.join(CHECKPOINT_DIR, f"checkpoint_{category}.jsonl")
    
    if not os.path.exists(checkpoint_path):
        return [], None
    
    examples = []
    try:
        with open(checkpoint_path, "r", encoding="utf-8") as f:
            for line in f:
                data = json.loads(line)
                # Reconstruct example based on category
                messages = [Message(role=m["role"], content=m["content"]) for m in data["messages"]]
                
                if category == "general":
                    example = GeneralExample(
                        id=data["id"],
                        category=Category(data["category"]),
                        messages=messages,
                        source_dataset=data.get("metadata", {}).get("source_dataset", ""),
                        original_id=data.get("metadata", {}).get("original_id", "")
                    )
                elif category == "identity":
                    example = IdentityExample(
                        id=data["id"],
                        category=Category(data["category"]),
                        messages=messages,
                        attack_type=data.get("metadata", {}).get("attack_type")
                    )
                elif category == "evidence":
                    example = EvidenceExample(
                        id=data["id"],
                        category=Category(data["category"]),
                        messages=messages,
                        context=data.get("metadata", {}).get("context", ""),
                        is_trap=data.get("metadata", {}).get("is_trap", False)
                    )
                elif category == "refusal":
                    example = RefusalExample(
                        id=data["id"],
                        category=Category(data["category"]),
                        messages=messages,
                        refusal_reason=data.get("metadata", {}).get("refusal_reason", "")
                    )
                elif category == "repair":
                    example = RepairExample(
                        id=data["id"],
                        category=Category(data["category"]),
                        messages=messages,
                        context=data.get("metadata", {}).get("context", ""),
                        hallucinated_claim=data.get("metadata", {}).get("hallucinated_claim", ""),
                        keyword_overlap=data.get("metadata", {}).get("keyword_overlap", 0.0)
                    )
                elif category == "contrast":
                    example = ContrastExample(
                        id=data["id"],
                        category=Category(data["category"]),
                        messages=messages,
                        variant_key=data.get("metadata", {}).get("variant_key", ""),
                        difference_type=data.get("metadata", {}).get("difference_type", "")
                    )
                elif category == "policy":
                    example = PolicyExample(
                        id=data["id"],
                        category=Category(data["category"]),
                        messages=messages,
                        scenario_type=data.get("metadata", {}).get("scenario_type", "approve"),
                        policy_context=data.get("metadata", {}).get("policy_context", "")
                    )
                else:
                    example = TrainingExample(
                        id=data["id"],
                        category=Category(data["category"]),
                        messages=messages
                    )
                
                examples.append(example)
        
        # Create stats from loaded checkpoint
        stats = GenerationStats(
            category=category,
            target_count=TARGET_COUNTS.get(category, 0),
            generated_count=len(examples)
        )
        
        print(f"  Loaded {len(examples)} examples from checkpoint")
        return examples, stats
        
    except Exception as e:
        print(f"  Error loading checkpoint: {e}")
        return [], None


def save_checkpoint(category: str, examples: List[TrainingExample]):
    """Save examples to checkpoint file."""
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    checkpoint_path = os.path.join(CHECKPOINT_DIR, f"checkpoint_{category}.jsonl")
    
    with open(checkpoint_path, "w", encoding="utf-8") as f:
        for example in examples:
            f.write(example.to_jsonl() + "\n")
    
    print(f"  Checkpoint saved: {checkpoint_path} ({len(examples)} examples)")


def print_discard_breakdown(stats: GenerationStats):
    """Print detailed discard breakdown for a category."""
    if stats.discarded_count == 0:
        print("  No discards")
        return
    
    print(f"  Discard breakdown ({stats.discarded_count} total):")
    if stats.json_invalid > 0:
        print(f"    - JSON invalid:       {stats.json_invalid:>5} ({stats.json_invalid/stats.discarded_count:>6.1%})")
    if stats.token_too_short > 0:
        print(f"    - Too short:          {stats.token_too_short:>5} ({stats.token_too_short/stats.discarded_count:>6.1%})")
    if stats.token_too_long > 0:
        print(f"    - Too long:           {stats.token_too_long:>5} ({stats.token_too_long/stats.discarded_count:>6.1%})")
    if stats.keyword_overlap_low > 0:
        print(f"    - Low keyword overlap:{stats.keyword_overlap_low:>5} ({stats.keyword_overlap_low/stats.discarded_count:>6.1%})")
    if stats.duplicate > 0:
        print(f"    - Duplicate:          {stats.duplicate:>5} ({stats.duplicate/stats.discarded_count:>6.1%})")
    if stats.other > 0:
        print(f"    - Other:              {stats.other:>5} ({stats.other/stats.discarded_count:>6.1%})")


class DatasetPipeline:
    """Main orchestrator for the dataset generation pipeline with checkpointing."""
    
    def __init__(self):
        self.all_examples: List[TrainingExample] = []
        self.category_stats: Dict[str, GenerationStats] = {}
        self.category_times: Dict[str, float] = {}
        self.start_time = None
        self.end_time = None
    
    def run(self) -> List[TrainingExample]:
        """Execute the full pipeline with checkpointing and resume."""
        self.start_time = time.time()
        
        print("\n" + "="*60)
        print("LUMIS-1 SPEAKER DATASET PIPELINE (PARALLEL + CHECKPOINTING)")
        print("="*60)
        print(f"Target: {sum(TARGET_COUNTS.values()):,} examples")
        print(f"Categories: {len(TARGET_COUNTS)}")
        print(f"Concurrent requests: {CONFIG['max_concurrent_requests']}")
        print(f"Checkpoint directory: {CHECKPOINT_DIR}/")
        print()
        
        # =====================================================
        # PHASE 1: OPEN DATASETS (General Category)
        # =====================================================
        print("#" * 60)
        print("# PHASE 1: OPEN DATASETS (General)")
        print("#" * 60)
        
        phase1_start = time.time()
        
        # Check for checkpoint
        print("\nChecking for checkpoint...")
        loaded_examples, loaded_stats = load_checkpoint("general")
        
        if loaded_examples and len(loaded_examples) >= TARGET_COUNTS["general"] * 0.9:
            print(f"  Using checkpoint ({len(loaded_examples)} examples)")
            general_examples = loaded_examples
            self.category_stats["general"] = loaded_stats
        else:
            print("  No valid checkpoint, generating fresh...")
            general_gen = GeneralGenerator(TARGET_COUNTS["general"])
            general_examples = general_gen.generate()
            self.category_stats["general"] = general_gen.stats
            
            # Save checkpoint
            save_checkpoint("general", general_examples)
            
            # Print discard breakdown
            print_discard_breakdown(general_gen.stats)
        
        self.all_examples.extend(general_examples)
        self.category_times["general"] = time.time() - phase1_start
        
        print(f"\nGeneral: {len(general_examples)} collected (target: {TARGET_COUNTS['general']})")
        print(f"Time: {self.category_times['general']:.1f}s")
        
        # =====================================================
        # PHASE 2: SYNTHETIC GENERATION (DeepSeek Teacher - PARALLEL)
        # =====================================================
        print("\n" + "#" * 60)
        print(f"# PHASE 2: SYNTHETIC GENERATION (DeepSeek - {CONFIG['max_concurrent_requests']} concurrent)")
        print("#" * 60)
        
        # Reset the semaphore for fresh event loop
        teacher_client._semaphore = None
        
        synthetic_generators = [
            ("identity", IdentityGenerator, TARGET_COUNTS["identity"]),
            ("evidence", EvidenceGenerator, TARGET_COUNTS["evidence"]),
            ("refusal", RefusalGenerator, TARGET_COUNTS["refusal"]),
            ("repair", RepairGenerator, TARGET_COUNTS["repair"]),
            ("contrast", ContrastGenerator, TARGET_COUNTS["contrast"]),
            ("policy", PolicyGenerator, TARGET_COUNTS["policy"]),
        ]
        
        for name, generator_class, target in synthetic_generators:
            print(f"\n{'='*60}")
            print(f"Generating {name.upper()} ({target} target)")
            print(f"{'='*60}")
            
            cat_start = time.time()
            
            # Check for checkpoint
            print("Checking for checkpoint...")
            loaded_examples, loaded_stats = load_checkpoint(name)
            
            if loaded_examples and len(loaded_examples) >= target * 0.9:
                print(f"  Using checkpoint ({len(loaded_examples)} examples)")
                examples = loaded_examples
                self.category_stats[name] = loaded_stats
                
                # Add loaded prompts to uniqueness gate
                for ex in loaded_examples:
                    for msg in ex.messages:
                        if msg.role == "user":
                            uniqueness_gate.seen_prompts.add(msg.content.strip().lower()[:200])
                            break
            else:
                print("  No valid checkpoint, generating fresh...")
                
                generator = generator_class(
                    client=teacher_client,
                    category=Category(name),
                    target_count=target
                )
                
                # Uses async parallelization internally
                examples = generator.generate_batch(show_progress=True)
                
                self.category_stats[name] = generator.stats
                
                # Save checkpoint immediately after completion
                save_checkpoint(name, examples)
                
                # Print discard breakdown
                print_discard_breakdown(generator.stats)
            
            cat_time = time.time() - cat_start
            self.category_times[name] = cat_time
            
            self.all_examples.extend(examples)
            
            # Summary for this category
            stats = self.category_stats[name]
            effective_rate = len(examples) / cat_time if cat_time > 0 else 0
            
            print(f"\n  Summary:")
            print(f"    Generated: {stats.generated_count}")
            print(f"    Discarded: {stats.discarded_count}")
            print(f"    Discard rate: {stats.discard_rate:.1%}")
            print(f"    Time: {cat_time:.1f}s ({effective_rate:.1f} examples/sec)")
        
        # =====================================================
        # FINALIZE
        # =====================================================
        self.end_time = time.time()
        runtime = self.end_time - self.start_time
        
        print("\n" + "="*60)
        print("PIPELINE COMPLETE")
        print("="*60)
        print(f"Total examples: {len(self.all_examples):,}")
        print(f"Total runtime: {runtime / 60:.1f} minutes ({runtime:.0f}s)")
        print(f"Average rate: {len(self.all_examples) / runtime:.1f} examples/sec")
        print(f"API calls: {teacher_client.total_calls}")
        print(f"API errors: {teacher_client.total_errors}")
        
        return self.all_examples
    
    def print_summary(self):
        """Print detailed summary statistics."""
        print("\n" + "="*60)
        print("GENERATION SUMMARY")
        print("="*60)
        print()
        print(f"{'Category':<12} {'Target':>8} {'Generated':>10} {'Discarded':>10} {'Rate':>8} {'Time':>8}")
        print("-" * 62)
        
        total_target = 0
        total_generated = 0
        total_discarded = 0
        
        for category in TARGET_COUNTS.keys():
            stats = self.category_stats.get(category)
            cat_time = self.category_times.get(category, 0)
            if stats:
                print(f"{category:<12} {stats.target_count:>8} {stats.generated_count:>10} "
                      f"{stats.discarded_count:>10} {stats.discard_rate:>7.1%} {cat_time:>7.1f}s")
                total_target += stats.target_count
                total_generated += stats.generated_count
                total_discarded += stats.discarded_count
        
        print("-" * 62)
        overall_rate = total_discarded / (total_generated + total_discarded) if (total_generated + total_discarded) > 0 else 0
        total_time = sum(self.category_times.values())
        print(f"{'TOTAL':<12} {total_target:>8} {total_generated:>10} {total_discarded:>10} {overall_rate:>7.1%} {total_time:>7.1f}s")
        
        # Discard breakdown by category
        print("\n" + "-"*60)
        print("DISCARD BREAKDOWN BY CATEGORY")
        print("-"*60)
        
        for category, stats in self.category_stats.items():
            if stats.discarded_count > 0:
                print(f"\n{category}:")
                if stats.json_invalid > 0:
                    print(f"  JSON invalid:        {stats.json_invalid}")
                if stats.token_too_short > 0:
                    print(f"  Too short:           {stats.token_too_short}")
                if stats.token_too_long > 0:
                    print(f"  Too long:            {stats.token_too_long}")
                if stats.keyword_overlap_low > 0:
                    print(f"  Low keyword overlap: {stats.keyword_overlap_low}")
                if stats.duplicate > 0:
                    print(f"  Duplicate:           {stats.duplicate}")
                if stats.other > 0:
                    print(f"  Other:               {stats.other}")
        
        # Performance summary
        if self.end_time and self.start_time:
            total_runtime = self.end_time - self.start_time
            print("\n" + "-"*60)
            print("PERFORMANCE")
            print("-"*60)
            print(f"  Total runtime: {total_runtime / 60:.1f} minutes")
            print(f"  Synthetic generation: {sum(self.category_times.get(k, 0) for k in ['identity', 'evidence', 'refusal', 'repair', 'contrast', 'policy']) / 60:.1f} minutes")
            print(f"  Throughput: {total_generated / total_runtime:.1f} examples/sec")
            print(f"  Speedup from parallelization: ~{CONFIG['max_concurrent_requests']}x vs sequential")
        
        # Checkpoint status
        print("\n" + "-"*60)
        print("CHECKPOINT STATUS")
        print("-"*60)
        if os.path.exists(CHECKPOINT_DIR):
            for cat in TARGET_COUNTS.keys():
                checkpoint_path = os.path.join(CHECKPOINT_DIR, f"checkpoint_{cat}.jsonl")
                if os.path.exists(checkpoint_path):
                    size = os.path.getsize(checkpoint_path)
                    print(f"  {cat}: {size / 1024:.1f} KB")
                else:
                    print(f"  {cat}: NOT FOUND")
        else:
            print(f"  Checkpoint directory not found: {CHECKPOINT_DIR}/")


# Run the pipeline
print("DatasetPipeline initialized with checkpointing and resume support.")
print(f"Expected runtime: ~{(8500 / (CONFIG['max_concurrent_requests'] * 1.5)) / 60:.0f}-{(8500 / (CONFIG['max_concurrent_requests'] * 0.8)) / 60:.0f} minutes (vs 14+ hours sequential)")
print()

pipeline = DatasetPipeline()
all_examples = pipeline.run()
pipeline.print_summary()

In [ ]:
# ============================================================================
# CELL 8: VALIDATION & EXPORT
# ============================================================================

def validate_dataset(examples: List[TrainingExample]) -> Dict:
    """Comprehensive validation of generated dataset."""
    stats = {
        "total_count": len(examples),
        "category_counts": {},
        "token_stats": {
            "min": float("inf"),
            "max": 0,
            "sum": 0,
        },
        "repair_overlap_stats": {
            "min": 1.0,
            "max": 0.0,
            "sum": 0.0,
            "count": 0,
        },
        "issues": []
    }
    
    # Count by category
    for cat in Category:
        stats["category_counts"][cat.value] = 0
    
    for example in examples:
        stats["category_counts"][example.category.value] += 1
        
        # Token stats
        tokens = example.token_count()
        stats["token_stats"]["min"] = min(stats["token_stats"]["min"], tokens)
        stats["token_stats"]["max"] = max(stats["token_stats"]["max"], tokens)
        stats["token_stats"]["sum"] += tokens
        
        # Repair overlap stats
        if isinstance(example, RepairExample) and example.keyword_overlap > 0:
            stats["repair_overlap_stats"]["min"] = min(
                stats["repair_overlap_stats"]["min"], example.keyword_overlap
            )
            stats["repair_overlap_stats"]["max"] = max(
                stats["repair_overlap_stats"]["max"], example.keyword_overlap
            )
            stats["repair_overlap_stats"]["sum"] += example.keyword_overlap
            stats["repair_overlap_stats"]["count"] += 1
    
    # Calculate averages
    if len(examples) > 0:
        stats["token_stats"]["mean"] = stats["token_stats"]["sum"] / len(examples)
    
    if stats["repair_overlap_stats"]["count"] > 0:
        stats["repair_overlap_stats"]["mean"] = (
            stats["repair_overlap_stats"]["sum"] / stats["repair_overlap_stats"]["count"]
        )
    
    # Check for issues
    for cat_name, target in TARGET_COUNTS.items():
        actual = stats["category_counts"].get(cat_name, 0)
        if actual < target * 0.90:  # Allow 10% variance
            stats["issues"].append(f"{cat_name}: {actual} < {target} (target)")
    
    return stats


def print_validation_report(stats: Dict):
    """Print formatted validation report."""
    print("\n" + "="*60)
    print("DATASET VALIDATION REPORT")
    print("="*60)
    
    print(f"\nTotal Examples: {stats['total_count']:,}")
    
    print("\nCategory Distribution:")
    print("-" * 45)
    print(f"{'Category':<12} {'Actual':>8} {'Target':>8} {'%':>8}")
    print("-" * 45)
    for cat_name, target in TARGET_COUNTS.items():
        actual = stats["category_counts"].get(cat_name, 0)
        pct = (actual / target * 100) if target > 0 else 0
        status = "OK" if pct >= 90 else "LOW"
        print(f"{cat_name:<12} {actual:>8} {target:>8} {pct:>7.1f}% {status}")
    
    print("\nToken Statistics:")
    print("-" * 30)
    ts = stats["token_stats"]
    print(f"  Min:  {ts['min']}")
    print(f"  Max:  {ts['max']}")
    print(f"  Mean: {ts.get('mean', 0):.1f}")
    
    rs = stats["repair_overlap_stats"]
    if rs["count"] > 0:
        print("\nRepair Keyword Overlap:")
        print("-" * 30)
        print(f"  Min:  {rs['min']:.1%}")
        print(f"  Max:  {rs['max']:.1%}")
        print(f"  Mean: {rs.get('mean', 0):.1%}")
        print(f"  Count: {rs['count']}")
    
    if stats["issues"]:
        print("\nISSUES FOUND:")
        print("-" * 30)
        for issue in stats["issues"]:
            print(f"  WARNING: {issue}")
    else:
        print("\nNo issues found.")


def export_dataset(
    examples: List[TrainingExample],
    train_path: str = "train.jsonl",
    eval_path: str = "eval.jsonl",
    eval_ratio: float = 0.10,
    seed: int = 42
):
    """Export dataset to train and eval JSONL files."""
    random.seed(seed)
    
    # Shuffle
    shuffled = examples.copy()
    random.shuffle(shuffled)
    
    # Split
    eval_count = int(len(shuffled) * eval_ratio)
    eval_examples = shuffled[:eval_count]
    train_examples = shuffled[eval_count:]
    
    print(f"\n{'='*60}")
    print("EXPORTING DATASET")
    print(f"{'='*60}")
    print(f"\nTrain: {len(train_examples):,} examples -> {train_path}")
    print(f"Eval:  {len(eval_examples):,} examples -> {eval_path}")
    
    # Write train
    with open(train_path, "w", encoding="utf-8") as f:
        for example in train_examples:
            f.write(example.to_jsonl() + "\n")
    
    # Write eval
    with open(eval_path, "w", encoding="utf-8") as f:
        for example in eval_examples:
            f.write(example.to_jsonl() + "\n")
    
    print("\nExport complete.")
    
    # Print sample
    print("\n" + "-"*60)
    print("SAMPLE TRAINING EXAMPLE")
    print("-"*60)
    sample = random.choice(train_examples)
    print(f"Category: {sample.category.value}")
    print(f"ID: {sample.id}")
    print(f"\nGranite format (first 500 chars):")
    print(sample.to_granite_format()[:500])
    if len(sample.to_granite_format()) > 500:
        print("...")
    
    return train_examples, eval_examples


# ============================================================================
# RUN VALIDATION & EXPORT
# ============================================================================

# Validate
print("\nValidating dataset...")
validation_stats = validate_dataset(all_examples)
print_validation_report(validation_stats)

# Export
train_data, eval_data = export_dataset(
    all_examples,
    train_path="train.jsonl",
    eval_path="eval.jsonl",
    eval_ratio=CONFIG["eval_ratio"],
    seed=CONFIG["seed"]
)

# Verify files
print("\n" + "="*60)
print("FILE VERIFICATION")
print("="*60)

import os
for path in ["train.jsonl", "eval.jsonl"]:
    if os.path.exists(path):
        size = os.path.getsize(path)
        with open(path, "r") as f:
            lines = sum(1 for _ in f)
        print(f"{path}: {lines:,} lines, {size / 1024 / 1024:.2f} MB")
    else:
        print(f"{path}: NOT FOUND")

print("\n" + "#"*60)
print("# LUMIS-1 SPEAKER DATASET GENERATION COMPLETE")
print("#"*60)